In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\ronal\source\repos\ARTEFACT-DesafioTecnico\.venv\Scripts\python.exe
3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [2]:
pip install -U sentence-transformers

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/611.3 kB ? eta -:--:--
   ---------------------------------------- 611.3/611.3 kB 10.3 MB/s  0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 8.3/8.3 MB 44.2 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [nar

In [3]:
def load_chunks(chunks_dir: Path) -> list[dict]:
    """
    Carrega todos os chunks dos arquivos JSONL.
    """

    chunks = []

    for jsonl_path in chunks_dir.glob("*.jsonl"):
        print(f"Carregando: {jsonl_path.name}")

        with jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                chunk = json.loads(line)
                chunks.append(chunk)

    return chunks

### Gera os embeddings

In [7]:
from pathlib import Path
import json
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")

chunks_dir = Path("data/chunks")

# Carrega todos os chunks dos JSONL
chunks = load_chunks(chunks_dir)

texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32,
)

print(embeddings.shape)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 28853.97it/s]


Carregando: politicas_da_loja.jsonl


Batches: 100%|██████████| 1/1 [00:07<00:00,  7.26s/it]

(8, 1024)


### Salva os embeddings

In [8]:
import numpy as np

np.save(
    "data/embeddings.npy",
    embeddings
)

### Inferência

In [11]:
import numpy as np

embeddings = np.load(
    "data/embeddings.npy"
)

query = "quais são as promoções disponíveis?"

query_embedding = model.encode(
    query,
    normalize_embeddings=True,
)

scores = embeddings @ query_embedding

top_k = np.argsort(scores)[::-1][:5]

for idx in top_k:
    print(scores[idx])
    print(chunks[idx]["source"])
    print(chunks[idx]["page"])
    #print(chunks[idx]["text"][:300])

0.63234645
politicas_da_loja.pdf
6
0.512078
politicas_da_loja.pdf
7
0.49520743
politicas_da_loja.pdf
8
0.48451513
politicas_da_loja.pdf
4
0.47480056
politicas_da_loja.pdf
3
